# 04 - 端到端流程（冒烟 / 全量 F1 最大化 / 全量 sid + 中断后续跑）

本笔记本是 **`docs/user_guide.md`** 中命令行流水线的交互版，支持 **断点标记中断后续跑**（日志在 `runs/pipeline-logs/`）。

**本笔记本侧重项目主评价指标：验证集 F1 最大化**（`early_stop_metric: best_f1` + `threshold_search.objective: f1`）。  
召回率优先 / 精度优先全量实验见 **`05_full_recall_vs_precision.ipynb`**。

## 三套流程

| 流程 | 断点标记前缀 | 实验 YAML | 说明 |
|------|-------------|-----------|------|
| **冒烟** | `smoke` | `exp_baseline.yaml` | ~500 行索引；**关闭 DDP** |
| **全量 F1 time** | `full_f1` | `exp_full_f1_ddp.yaml` | 全量索引 + **time_stratified**；**F1 最大化** |
| **全量 F1 sid** | `full_f1_sid` | `exp_full_f1_sid.yaml` | **共用 processed/**；`resplit.sh` → 跨机泛化 |

**推荐顺序**：冒烟 → 全量 F1 time（含 preprocess）→ **全量 F1 sid**（仅 resplit + 训练，不重复扫 mcelog）。

## 输出与指标

- 训练/推理：`runs/exp_tianchi_full_f1_ddp/` 与 `runs/exp_tianchi_full_f1_sid/`
- **部署阶段评价指标**：`metrics.json` 顶层 / `deployment`（验证集 **F1 最优阈值**）；**不得**将测试集事后最优评价作为部署性能承诺
- 正式指标见 `outputs/tables/full_f1_time_vs_sid_deployment_kpi.csv`
- 权重：`runs/<run_name>/<train_session>/checkpoints/grid_cnn.pt`（可用 `latest_train`）
- 误差分析：`03_error_analysis.ipynb`（FN/FP × `empty_mcelog_window` × sid）；全量主实验混淆见 `outputs/tables/error_confusion_counts_full_f1.csv`

## 使用提示

- 断点标记前缀 **≠** `run_name`（例：`full_f1_sid` vs `exp_tianchi_full_f1_sid`）
- sid 单元要求实验 YAML 中 `processed_dir/preprocess_manifest.json` 存在（先跑全量 time 的 preprocess）
- 改索引/horizon：清空 `data/processed/*` 与各 `splits*` 并删相关 marker
- **源码更新后 import 报错**：先运行下方 setup 单元，或 **Kernel → Restart** 后从头执行

命令行：冒烟 **`bash scripts/autotune.sh`**；全量 F1 **`exp_full_f1_*.yaml`**；sid **`bash scripts/resplit.sh`**。

**全量开关**：未设置 `DRAM2CNN_RUN_FULL_NOTEBOOKS=1` 时，下方全量 F1 / sid 单元会跳过；冒烟单元始终执行。


In [ ]:
import os
from pathlib import Path

from dram2cnn.utils.notebook_bootstrap import notebook_setup
from dram2cnn.utils.pipeline_helpers import pipeline_log_dir, pipeline_marker_dir

PROJECT_ROOT = notebook_setup(cwd=Path.cwd().resolve())
LOG_DIR = pipeline_log_dir(PROJECT_ROOT)
MARKER_DIR = pipeline_marker_dir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOG_DIR:", LOG_DIR)
print("MARKER_DIR:", MARKER_DIR)


In [ ]:
from dram2cnn.utils.pipeline_helpers import (
    clear_exp_markers,
    preflight_full_scale_index,
    preflight_split_labels,
    preflight_splits_vs_model,
    require_preprocess_manifest,
    run_step,
)
from dram2cnn.utils.deployment_kpi import (
    F1_KPI_PAIRS,
    collect_kpi_rows,
    merge_all_strategies_kpi,
    write_kpi_csv,
)

# 实验 YAML / 断点标记前缀（全 notebook 共用，避免后续单元 NameError）
SMOKE_EXP = "smoke"
FULL_EXP = "full_f1"
SID_EXP = "full_f1_sid"
SMOKE_CFG = "configs/experiments/exp_baseline.yaml"
FULL_CFG = "configs/experiments/exp_full_f1_ddp.yaml"
SID_CFG = "configs/experiments/exp_full_f1_sid.yaml"

print("pipeline helpers loaded (run_step, preflight, deployment_kpi, 实验常量)")

# 全量单元开关：默认关闭，避免误跑数小时全量训练。
# 开启方式：环境变量 DRAM2CNN_RUN_FULL_NOTEBOOKS=1，或本格改为 True 后重新执行。
import os as _os
RUN_FULL_SCALE = _os.environ.get("DRAM2CNN_RUN_FULL_NOTEBOOKS", "0").strip() in {"1", "true", "True", "YES", "yes"}
print("RUN_FULL_SCALE:", RUN_FULL_SCALE, "（工程实验/全量默认跳过；设 DRAM2CNN_RUN_FULL_NOTEBOOKS=1 后重新执行）")


---
## 冒烟测试（Smoke）

目标：快速验证 **索引 → 预处理 → 训练 → 推理 → 评估** 全链路可用。

| 步骤 | 说明 |
|------|------|
| 1. 生成演示索引 | `features_5m_index_demo.csv`（~500 行） |
| 2. preprocess | `exp_baseline.yaml`，截断 mcelog/索引 |
| 3. train | `train.sh`（exp_baseline **关闭 DDP** 以加速） |
| 4. infer / evaluate | 推理与指标 |

命令行自动化等价（无断点标记）：**`bash scripts/autotune.sh`** = 预处理（含校验）→ 训练 → 推理 → 评估（**不含** build_index/inspect_data；完整冒烟见本笔记本冒烟单元）。

In [ ]:
# 【冒烟 1/2】重建演示索引（与 configs/data.yaml 默认 index_features_path 一致）
run_step(SMOKE_EXP, "build_index", "bash scripts/build_index.sh smoke")

In [ ]:
# 【冒烟 2/2】baseline 全流程（configs/experiments/exp_baseline.yaml）

run_step(SMOKE_EXP, "preprocess", f"bash scripts/preprocess.sh {SMOKE_CFG}")
# preprocess.sh 末尾已 validate；以下为可选重复诊断
# run_step(SMOKE_EXP, "validate", f"bash scripts/validate.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "inspect_data", f"bash scripts/inspect_data.sh {SMOKE_CFG}")

preflight_splits_vs_model(SMOKE_CFG, PROJECT_ROOT)
preflight_split_labels(SMOKE_CFG, PROJECT_ROOT)

run_step(SMOKE_EXP, "train", f"bash scripts/train.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "infer", f"bash scripts/infer.sh {SMOKE_CFG}")
run_step(SMOKE_EXP, "evaluate", f"bash scripts/evaluate.sh {SMOKE_CFG}")

print("冒烟完成 → runs/exp_tianchi_baseline/")
print("后续分析：02_feature_analysis.ipynb（X 分布）、03_error_analysis.ipynb（FP/FN，EXP 仍用 exp_baseline）")

In [5]:
# 【可选】命令行 autotune 端到端链路检查（不分 marker；与上一单元二选一即可）
# run_step(SMOKE_EXP, "autotune", "bash scripts/autotune.sh")

---
## 全量 F1 最大化（全量 + 多卡）

目标：在 **真实全量索引 + 完整 mcelog** 上训练，**早停与阈值搜索均最大化验证集 F1**（项目主评价指标定义）。

| 步骤 | 说明 |
|------|------|
| 1. 生成全量索引 | 输出 `data/raw/features_index_h300s_step300s_full.csv`（耗时长） |
| 2. **配置** | `exp_full_f1_ddp.yaml`；`drop_empty_positive_windows: false`（与历史 F1 基线一致） |
| 3. 清空旧输出 | 若曾跑工程实验预处理：`rm -rf data/processed/* data/splits/*` 及 `full_f1__preprocess.done` |
| 4. preprocess → evaluate | **`train_ddp.sh`** 多卡；输出 `runs/exp_tianchi_full_f1_ddp/` |

> 终端等价：`bash scripts/preprocess.sh configs/experiments/exp_full_f1_ddp.yaml`


In [ ]:
if not RUN_FULL_SCALE:
    print("[SKIP] 全量单元未启用（RUN_FULL_SCALE=False）。冒烟路径已足够验证链路；全量请设 DRAM2CNN_RUN_FULL_NOTEBOOKS=1。")
else:
    # 【全量 F1 1/3】扫描完整 mcelog 生成全量索引（耗时长，可跳过若 CSV 已存在且断点标记已存在）
    run_step(FULL_EXP, "build_index", "bash scripts/build_index.sh full")

    print(
        "\n下一步：\n"
        "  1) exp_full_f1_ddp.yaml → index_features_path=data/raw/features_index_h300s_step300s_full.csv\n"
        "  2) 若曾跑过工程实验预处理：rm -rf data/processed/* data/splits/*\n"
        "  3) clear_exp_markers('full_f1')  # 或删 full_f1__preprocess.done\n"
        "  4) 运行下方「全量 F1 预先检查 + 流水线」单元"
    )


In [ ]:
if not RUN_FULL_SCALE:
    print("[SKIP] 全量单元未启用（RUN_FULL_SCALE=False）。冒烟路径已足够验证链路；全量请设 DRAM2CNN_RUN_FULL_NOTEBOOKS=1。")
else:
    # 【全量 F1 2/3】预先检查：index_features_path 不得仍指向演示 stub
    _cfg = preflight_full_scale_index(FULL_CFG, root=PROJECT_ROOT, context="04_full_f1_preflight")
    print(f"[preflight] 全量 index_features_path OK: {_cfg['data']['index_features_path']}")
    print(f"[preflight] early_stop_metric={_cfg['train']['early_stop_metric']} threshold_objective={_cfg['train']['threshold_search']['objective']}")


In [ ]:
if not RUN_FULL_SCALE:
    print("[SKIP] 全量单元未启用（RUN_FULL_SCALE=False）。冒烟路径已足够验证链路；全量请设 DRAM2CNN_RUN_FULL_NOTEBOOKS=1。")
else:
    # 【全量 F1 3/3】exp_full_f1_ddp：preprocess → DDP 训练 → 推理 → 评估
    run_step(FULL_EXP, "preprocess", f"bash scripts/preprocess.sh {FULL_CFG}")
    run_step(FULL_EXP, "inspect_data", f"bash scripts/inspect_data.sh {FULL_CFG}")

    preflight_splits_vs_model(FULL_CFG, PROJECT_ROOT)
    preflight_split_labels(FULL_CFG, PROJECT_ROOT)

    run_step(FULL_EXP, "train", f"bash scripts/train_ddp.sh {FULL_CFG}")
    run_step(FULL_EXP, "infer", f"bash scripts/infer.sh {FULL_CFG}")
    run_step(FULL_EXP, "evaluate", f"bash scripts/evaluate.sh {FULL_CFG}")

    print("全量 F1 time 完成 → runs/exp_tianchi_full_f1_ddp/")
    print("分析 03 时请将 EXP_YAML 改为 exp_full_f1_ddp.yaml")


---
## 全量 F1 sid 泛化对比

目标：在**同一** `processed_dir` 上按 **sid** 切分，与 **time_stratified**（`full_f1`）对比 **部署阶段 F1**。

| 步骤 | 说明 |
|------|------|
| 0. 前置 | 已完成 **全量 F1 time** 的 `preprocess` |
| 1. resplit → evaluate | 输出 `runs/exp_tianchi_full_f1_sid/` |

命令行等价：**`bash scripts/resplit.sh configs/experiments/exp_full_f1_sid.yaml`**

In [ ]:
if not RUN_FULL_SCALE:
    print("[SKIP] 全量单元未启用（RUN_FULL_SCALE=False）。冒烟路径已足够验证链路；全量请设 DRAM2CNN_RUN_FULL_NOTEBOOKS=1。")
else:
    # 【F1 sid 1/2】resplit → 训练 → 推理 → 评估

    _proc_man = require_preprocess_manifest(
        FULL_CFG,
        root=PROJECT_ROOT,
        hint_cmd=f"bash scripts/preprocess.sh {FULL_CFG}",
    )
    print("[preflight] processed OK:", _proc_man)

    run_step(SID_EXP, "resplit", f"bash scripts/resplit.sh {SID_CFG}")
    run_step(SID_EXP, "inspect_data", f"bash scripts/inspect_data.sh {SID_CFG}")

    preflight_splits_vs_model(SID_CFG, PROJECT_ROOT)
    preflight_split_labels(SID_CFG, PROJECT_ROOT)

    run_step(SID_EXP, "train", f"bash scripts/train_ddp.sh {SID_CFG}")
    run_step(SID_EXP, "infer", f"bash scripts/infer.sh {SID_CFG}")
    run_step(SID_EXP, "evaluate", f"bash scripts/evaluate.sh {SID_CFG}")

    print("全量 F1 sid 完成 → runs/exp_tianchi_full_f1_sid/")
    print("分析：03_error_analysis.ipynb，EXP_YAML=exp_full_f1_sid.yaml")


In [ ]:
if not RUN_FULL_SCALE:
    print("[SKIP] 全量单元未启用（RUN_FULL_SCALE=False）。冒烟路径已足够验证链路；全量请设 DRAM2CNN_RUN_FULL_NOTEBOOKS=1。")
else:
    # 【F1 sid 2/2】time vs sid：部署阶段评价指标对比
    rows = collect_kpi_rows(F1_KPI_PAIRS, project_root_path=PROJECT_ROOT)
    if rows:
        import pandas as pd

        cmp_df = pd.DataFrame(rows)
        print(cmp_df.to_string(index=False))
        out_cmp = PROJECT_ROOT / "outputs" / "tables"
        write_kpi_csv(rows, out_cmp / "full_f1_time_vs_sid_deployment_kpi.csv")
        merged = merge_all_strategies_kpi(tables_dir=out_cmp)
        print("\n已写入 outputs/tables/full_f1_time_vs_sid_deployment_kpi.csv")
        if merged:
            print(f"已更新汇总表 {merged.name}")


In [ ]:
# 查看最近日志与 marker
print("\n=== 最近日志（最多 15 条）===")
for p in sorted(LOG_DIR.glob("*.log"), key=lambda x: x.stat().st_mtime, reverse=True)[:15]:
    print(p.name)

print("\n=== 已完成的断点标记 ===")
for p in sorted(MARKER_DIR.glob("*.done")):
    print(p.name, "=>", p.read_text(encoding="utf-8").strip())

In [ ]:
# 重置 marker（可选 — 取消注释后再运行本格）
# clear_exp_markers("smoke")
# clear_exp_markers("full_f1")
# clear_exp_markers("full_f1_sid")
print("未删除 marker。需重跑某流程时取消上方 clear_exp_markers(...) 注释。")